In [18]:
!wget https://raw.githubusercontent.com/langchain-ai/langchain/7ab4a7841a2fe77e3b305c876e3e1eb31d2a9fce/docs/docs/example_data/nke-10k-2023.pdf -O ./tmp/nke-10k-2023.pdf

--2025-12-04 23:40:36--  https://raw.githubusercontent.com/langchain-ai/langchain/7ab4a7841a2fe77e3b305c876e3e1eb31d2a9fce/docs/docs/example_data/nke-10k-2023.pdf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


200 OK
Length: 2397936 (2.3M) [application/octet-stream]
Saving to: ‘./tmp/nke-10k-2023.pdf’

./tmp/nke-10k-2023. 100%[===================>]   2.29M  4.82MB/s    in 0.5s    

2025-12-04 23:40:37 (4.82 MB/s) - ‘./tmp/nke-10k-2023.pdf’ saved [2397936/2397936]



In [12]:
!uv pip install -qU onnxruntime

In [1]:
# custom converter settings
import os
from typing import Optional, Any, Iterator, AsyncIterator, Union
import logging
import traceback
from langchain_core.documents import Document
from langchain_core.document_loaders import BaseLoader
from langchain_core.runnables import run_in_executor
from docling.document_converter import DocumentConverter, InputFormat, PdfFormatOption, ImageFormatOption
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.pipeline_options import TableStructureOptions, TableFormerMode, RapidOcrOptions
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableStructureOptions, TableFormerMode
def _doclingConverter() -> DocumentConverter:
  _pipeline_config = {
      "accelerator_options": AcceleratorOptions(
          device=AcceleratorDevice.AUTO,
          cuda_use_flash_attention2=True,
      ),
      "table_structure_options": TableStructureOptions(mode=TableFormerMode.ACCURATE),
  }
  _base_pipeline_options = PdfPipelineOptions(
      **_pipeline_config,
      do_ocr=False)  
  _ocr_pipeline_options = PdfPipelineOptions(
      **_pipeline_config,
      ocr_options=RapidOcrOptions(
         print_verbose=False,         
         text_score=0.5,
         #rapidocr_params={"det_use_cuda": True}
         ))  
  doc_converter = DocumentConverter(
      format_options={          
          InputFormat.PDF: PdfFormatOption(
              pipeline_options=_base_pipeline_options,
          ),
          InputFormat.IMAGE: ImageFormatOption(
              pipeline_options=_ocr_pipeline_options,
          ),
      }
  )
  for frm in [InputFormat.PDF, InputFormat.IMAGE]:
     doc_converter.initialize_pipeline(frm)
  return doc_converter

class DoclingLoader(BaseLoader):
  _doc_converter: Optional[DocumentConverter] = None
  def __init__(self, file_path: str | list[str], **kwargs: Any) -> None:
      self._file_paths = file_path if isinstance(file_path, list) else [file_path]
      if DoclingLoader._doc_converter is None:
          DoclingLoader._doc_converter = _doclingConverter()
      self._converter = DoclingLoader._doc_converter
      self._kwargs = kwargs
  def load(self) -> list[Document]:
      """Load data into Document objects."""
      return list(self.lazy_load())
  async def aload(self) -> list[Document]:
      """Load data into Document objects."""
      return [document async for document in self.alazy_load()]
  async def alazy_load(self) -> AsyncIterator[Document]:
      """A lazy loader for Documents."""
      iterator = await run_in_executor(None, self.lazy_load)
      done = object()
      while True:
          doc = await run_in_executor(None, next, iterator, done)  # type: ignore[call-arg, arg-type]
          if doc is done:
              break
          yield doc  # type: ignore[misc]
  def lazy_load(self) -> Iterator[Document]:
      for source in self._file_paths:
            _result = self._converter.convert(
            os.path.abspath(source),
            raises_on_error=True)
            doc = _result.document
            text = doc.export_to_markdown(image_placeholder="")
            yield Document(page_content=text, metadata={"source": source})

In [3]:
docs = []
# Load documents from a local and a remote pdf file
#for file in [os.path.abspath("./tmp/nke-10k-2023.pdf"), "https://arxiv.org/pdf/2408.09869"]:
for file in ["./tmp/nke-10k-2023.pdf"]:
    loader = DoclingLoader(file_path=file)
    docs.extend(loader.load())
    print(f"Loaded {len(docs)} documents from {file}")
print(docs[0].page_content[:100])  

2025-12-05 11:25:01,326 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-12-05 11:25:01,389 - INFO - Going to convert document batch...
2025-12-05 11:25:01,390 - INFO - Processing document nke-10k-2023.pdf
2025-12-05 11:25:32,899 - INFO - Finished converting document nke-10k-2023.pdf in 31.58 sec.


Loaded 1 documents from ./tmp/nke-10k-2023.pdf
Class B Common Stock

(Title of each class)

Oregon

(State or other jurisdiction of incorporation)



In [1]:
!uv pip install -qU 'markitdown[all]'

In [28]:
# rely on LLM to ingest documents
from markitdown import MarkItDown
from openai import OpenAI

file = "../02/iris.jpg"
#file = "https://arxiv.org/pdf/2408.09869"

print("---- 1. Without LLM ----")
# rely on exiftool, if installed: `exiftool -ver`
# macOS: brew install exiftool
# Windows: winget install exiftool
# linux: sudo apt-get install libimage-exiftool-perl
md = MarkItDown(enable_builtins=True)
result = md.convert(file)
print(result.text_content)

print("---- 2. Using DoclingLoader ----")
doc =  DoclingLoader(file_path=file).load()
print(doc[0].page_content)

print("---- 3. Using LLM ----")
client = OpenAI()
md = MarkItDown(llm_client=client, llm_model="gpt-4o")
result = md.convert(file)
print(result.text_content)

print("---- 4. Using local LLM ----")
client = OpenAI(base_url="http://localhost:11434/v1/", api_key="sk-dummy-key")
md = MarkItDown(llm_client=client, llm_model="qwen3-vl:8b")
result = md.convert(file)
print(result.text_content)


2025-12-07 12:26:55,094 - INFO - detected formats: [<InputFormat.IMAGE: 'image'>]
2025-12-07 12:26:55,099 - INFO - Going to convert document batch...
2025-12-07 12:26:55,099 - INFO - Processing document iris.jpg


---- 1. Without LLM ----
ImageSize: 980x980

---- 2. Using DoclingLoader ----


2025-12-07 12:26:56,169 - INFO - Finished converting document iris.jpg in 1.07 sec.


## STRUCTUREOFAFLOWER
---- 3. Using LLM ----


2025-12-07 12:26:59,507 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


ImageSize: 980x980

# Description:
This detailed diagram illustrates the structure of a flowering plant. At the top, it showcases the essential reproductive components: the stamen and pistil. The stamen, marked with a male symbol, consists of the anther and filament. The pistil, denoted with a female symbol, comprises the stigma, style, ovary, and ovules, which are crucial for pollination and seed development.

Surrounding these reproductive parts are the brightly colored petals, designed to attract pollinators. Below, the sepals provide protection to the budding flower. The base of the flower is labeled with the receptacle and pedicel, supporting and connecting it to the stem. This comprehensive visualization serves as an educational tool for understanding floral anatomy and function.

---- 4. Using local LLM ----


2025-12-07 12:27:31,157 - INFO - HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"


ImageSize: 980x980

# Description:
This educational infographic titled **"STRUCTURE OF A FLOWER"** (in bold, uppercase orange text) provides a clear, cross-sectional illustration of a typical flower’s anatomy, designed for botanical or biology education. The central image features a **vibrant orange flower** with smooth, rounded petals that transition from deep orange at the edges to a softer hue near the center, creating a polished, modern aesthetic against a clean white background.

At the heart of the flower, **stamens (male reproductive organs, marked with a male symbol ♂)** are prominently displayed: long, slender **filaments** (bright yellow stalks) extend upward, each topped with an **anther** (small, light beige oval structures containing pollen). These stamens are arranged symmetrically, with arrows pointing precisely to each labeled part for clarity.

To the right, the **pistil (female reproductive organ, marked with a female symbol ♀)** is highlighted. It consists of a **sti

In [10]:
# directory loader
import os

def is_local_file(file_path: str) -> bool:
    return os.path.isfile(file_path)
def kb_folder() -> str:
    return os.path.abspath("./tmp/kb/")

#empty kb folder
if os.path.exists(kb_folder()):
    import shutil
    shutil.rmtree(kb_folder())

# Load documents from a local and a remote pdf file
for file in ["../02/medical-costs.csv", "../02/Kleiber-law.xlsx", "../02/iris.jpg", "../01/karpathy-x-english.png", "https://arxiv.org/pdf/2408.09869"]:
    if is_local_file(file):
        #move the file to tmp/src folder
        with open(file, "rb") as fsrc:
            os.makedirs(kb_folder(), exist_ok=True)
            with open(os.path.join(kb_folder(), os.path.basename(file)), "wb") as fdst:
                fdst.write(fsrc.read())
    else:
        #download the file to tmp/src folder
        import requests
        response = requests.get(file)
        os.makedirs(kb_folder(), exist_ok=True)
        with open(os.path.join(kb_folder(), os.path.basename(file)), "wb") as fdst:
            fdst.write(response.content)

# Now load all the files from the kb_folder using DirectoryLoader
from langchain_community.document_loaders import DirectoryLoader
loader = DirectoryLoader(kb_folder(), glob="**/*", use_multithreading=True, show_progress=True, loader_cls=DoclingLoader)
docs = await loader.aload()
print(f"Loaded {len(docs)} documents from folder {kb_folder()}")


  0%|          | 0/5 [00:00<?, ?it/s]2025-12-05 11:32:24,883 - INFO - detected formats: [<InputFormat.IMAGE: 'image'>]
2025-12-05 11:32:24,884 - INFO - detected formats: [<InputFormat.XLSX: 'xlsx'>]
2025-12-05 11:32:24,885 - INFO - detected formats: [<InputFormat.CSV: 'csv'>]
2025-12-05 11:32:24,886 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-12-05 11:32:24,887 - INFO - Going to convert document batch...
2025-12-05 11:32:24,888 - INFO - Initializing pipeline for SimplePipeline with options hash 995a146ad601044538e6a923bea22f4e
2025-12-05 11:32:24,890 - INFO - Processing document medical-costs.csv
2025-12-05 11:32:24,891 - INFO - Parsing CSV with delimiter: ","
2025-12-05 11:32:24,892 - INFO - Detected 1339 lines
2025-12-05 11:32:24,924 - INFO - Finished converting document medical-costs.csv in 0.04 sec.
 20%|██        | 1/5 [00:00<00:00,  8.03it/s]2025-12-05 11:32:25,011 - INFO - Going to convert document batch...
2025-12-05 11:32:25,013 - INFO - detected formats: [<Inpu

Loaded 5 documents from folder /home/admin/_github/massimodipaolo/ai-crash-course/07/tmp/kb


In [11]:
import markdown
from IPython.core.display import HTML
for doc in docs:
    print(f"\n----- Document {doc.metadata.get('source', 'unknown')} -----")
    _content = markdown.markdown(doc.page_content)
    display(HTML(_content[:200]))    


----- Document /home/admin/_github/massimodipaolo/ai-crash-course/07/tmp/kb/medical-costs.csv -----



----- Document /home/admin/_github/massimodipaolo/ai-crash-course/07/tmp/kb/Kleiber-law.xlsx -----



----- Document /home/admin/_github/massimodipaolo/ai-crash-course/07/tmp/kb/iris.jpg -----



----- Document /home/admin/_github/massimodipaolo/ai-crash-course/07/tmp/kb/karpathy-x-english.png -----



----- Document /home/admin/_github/massimodipaolo/ai-crash-course/07/tmp/kb/2408.09869 -----


In [6]:
!uv pip install langchain_huggingface

Using Python 3.12.3 environment at: /home/admin/_github/massimodipaolo/ai-crash-course/.venv
Audited 1 package in 5ms


In [ ]:
import time
from datetime import datetime
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

def log_time(step):
    ts = datetime.now().strftime('%H:%M:%S')
    print(f"[{ts}] {step}")

log_time("START: Building RAG system...")

# Step 1: Text splitting
start = time.time()
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=3_000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)
log_time(f"✓ Text splitting: {time.time() - start:.2f}s ({len(all_splits)} chunks)")

# Step 2: Embeddings 
start = time.time()
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L12-v2",  # Fast and lightweight
    model_kwargs={'device': device}  
)
log_time(f"✓ Embeddings model loaded: {time.time() - start:.2f}s")

start = time.time()
vector_store = InMemoryVectorStore.from_documents(all_splits, embeddings)
log_time(f"✓ Vector store created: {time.time() - start:.2f}s")
log_time(f"   → Embedded {len(all_splits)} chunks")

# Step 3: LLM setup
start = time.time()
from langchain.chat_models import init_chat_model
model = "granite4:3b-h"
llm = init_chat_model(
    model_provider="ollama",
    model=model,
    # kwargs passed to the model:
    temperature=0,
    timeout=30,
    max_tokens=1000,
)
log_time(f"✓ LLM initialized: {time.time() - start:.2f}s")

# Step 4: Tool and agent setup
start = time.time()
from langchain.agents import create_agent
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=5)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

prompt = (
    "You have access to a tool that retrieves context from a pdf documents. "
    "Use the tool to help answer user queries."
)
agent = create_agent(llm, [retrieve_context], system_prompt=prompt)
log_time(f"✓ Agent created: {time.time() - start:.2f}s")

# Step 5: Agent execution
start = time.time()
messages = [{"role": "user", "content": "How many OCR are available in Docling?"}]
_rs = agent.invoke({"messages": messages})
log_time(f"✓ Agent execution: {time.time() - start:.2f}s")

log_time("COMPLETE!")

# Display results
import markdown
from IPython.core.display import HTML
for msg in _rs['messages']:
    _content = markdown.markdown(msg.content)
    display(HTML(_content[:500]))

2025-12-05 11:33:23,392 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L12-v2


Using device: cuda
[11:33:23] START: Building RAG system...
[11:33:23] ✓ Text splitting: 0.00s (55 chunks)
[11:33:25] ✓ Embeddings model loaded: 2.44s
[11:33:25] ✓ Vector store created: 0.07s
[11:33:25]    → Embedded 55 chunks
[11:33:25] ✓ LLM initialized: 0.05s
[11:33:25] ✓ Agent created: 0.00s


2025-12-05 11:33:26,379 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2025-12-05 11:33:27,282 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[11:33:28] ✓ Agent execution: 2.83s
[11:33:28] COMPLETE!


In [30]:
!uv pip install markdownify

Using Python 3.12.3 environment at: /home/admin/_github/massimodipaolo/ai-crash-course/.venv
⠙ markdownify==1.2.2                                                            

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Resolved 5 packages in 167ms                                         
⠙ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)-------------------     0 B/15.36 KiB           
⠙ Preparing packages... (0/1)---------- 15.36 KiB/15.36 KiB         
Prepared 1 package in 22ms                                                        
Installed 1 package in 2ms                                  
 + markdownify==1.2.2


In [19]:
import nest_asyncio
nest_asyncio.apply()
import os
os.environ["USER_AGENT "] = "ai-crash-course-spider-1.0"
from langchain_community.document_loaders.sitemap import SitemapLoader
from langchain_community.document_transformers import MarkdownifyTransformer as markdownify
from bs4 import BeautifulSoup


def remove_nav_and_header_elements(content: BeautifulSoup) -> str:
    # Find all 'nav' and 'header' elements in the BeautifulSoup object
    nav_elements = content.find_all("nav")
    header_elements = content.find_all("header")
    # Remove each 'nav' and 'header' element from the BeautifulSoup object
    for element in nav_elements + header_elements:
        element.decompose()
    return str(content.get_text())
async def alazy_load(loader: SitemapLoader) -> AsyncIterator[Document]:
    """A lazy loader for Documents."""
    iterator = await run_in_executor(None, loader.lazy_load)
    done = object()
    while True:
        doc = await run_in_executor(None, next, iterator, done)  # type: ignore[call-arg, arg-type]
        if doc is done:
            break
        yield doc  # type: ignore[misc]

def _output(documents: list[Document]) -> list[Document]:
    return list(markdownify().transform_documents(documents)) 

sitemap_loader = SitemapLoader(
    web_path="https://reference.langchain.com/python/sitemap.xml",
    filter_urls=["https://reference.langchain.com/python/langchain_core"],
    parsing_function=remove_nav_and_header_elements,)
docs = _output([document async for document in alazy_load(sitemap_loader)])


Fetching pages: 100%|##########| 16/16 [00:02<00:00,  6.82it/s]


In [30]:
import markdown
import random
from IPython.core.display import HTML
for doc in random.sample(docs, min(5, 5)):
    print(f"{doc.page_content[:300]}")
    print(f"----\n")
    


Retrievers | LangChain Reference
Skip to content
Retrievers
BaseRetriever
¶
Bases: RunnableSerializable[RetrieverInput, RetrieverOutput], ABC
Abstract base class for a document retrieval system.
A retrieval system is defined as something that can take string queries and return
the most 'relevant' do
----

Serialization | LangChain Reference
Skip to content
Serialization
dumpd
¶
dumpd(obj: Any) -> Any
Return a dict representation of an object.
PARAMETER
DESCRIPTION
obj
The object to dump.
TYPE:
Any
RETURNS
DESCRIPTION
Any
Dictionary that can be serialized to json using json.dumps.
dumps
¶
dumps(obj: A
----

Document loaders | LangChain Reference
Skip to content
Document loaders
document\_loaders
¶
Document loaders.
BaseLoader
¶
Bases: ABC
Interface for Document Loader.
Implementations should implement the lazy-loading method using generators
to avoid loading all documents into memory at once.
load is p
----

Output parsers | LangChain Reference
Skip to content
Output parsers
output\_par

In [31]:
import time
from datetime import datetime
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

def log_time(step):
    ts = datetime.now().strftime('%H:%M:%S')
    print(f"[{ts}] {step}")

log_time("START: Building RAG system...")

# Step 1: Text splitting
start = time.time()
from langchain_text_splitters import MarkdownTextSplitter
text_splitter = MarkdownTextSplitter(chunk_size=3_000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)
log_time(f"✓ Text splitting: {time.time() - start:.2f}s ({len(all_splits)} chunks)")

# Step 2: Embeddings 
start = time.time()
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L12-v2",  # Fast and lightweight
    model_kwargs={'device': device}  
)
log_time(f"✓ Embeddings model loaded: {time.time() - start:.2f}s")

start = time.time()
vector_store = InMemoryVectorStore.from_documents(all_splits, embeddings)
log_time(f"✓ Vector store created: {time.time() - start:.2f}s")
log_time(f"   → Embedded {len(all_splits)} chunks")

# Step 3: LLM setup
start = time.time()
from langchain.chat_models import init_chat_model
model = "granite4:3b-h"
llm = init_chat_model(
    model_provider="ollama",
    model=model,
    # kwargs passed to the model:
    temperature=0,
    timeout=30,
    max_tokens=1000,
)
log_time(f"✓ LLM initialized: {time.time() - start:.2f}s")

# Step 4: Tool and agent setup
start = time.time()
from langchain.agents import create_agent
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=5)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

prompt = (
    "You have access to a tool that retrieves context from a stored kb. "
    "Use the tool to help answer user queries."
)
agent = create_agent(llm, [retrieve_context], system_prompt=prompt)
log_time(f"✓ Agent created: {time.time() - start:.2f}s")

# Step 5: Agent execution
start = time.time()
messages = [{"role": "user", "content": "How can I use LangChain to load data from a sitemap?"}]
_rs = agent.invoke({"messages": messages})
log_time(f"✓ Agent execution: {time.time() - start:.2f}s")

log_time("COMPLETE!")

# Display results
import markdown
from IPython.core.display import HTML
for msg in _rs['messages']:
    _content = markdown.markdown(msg.content)
    display(HTML(_content))

2025-12-05 17:58:17,345 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L12-v2


Using device: cuda
[17:58:17] START: Building RAG system...
[17:58:17] ✓ Text splitting: 0.02s (449 chunks)
[17:58:19] ✓ Embeddings model loaded: 2.48s
[17:58:20] ✓ Vector store created: 0.54s
[17:58:20]    → Embedded 449 chunks
[17:58:20] ✓ LLM initialized: 0.03s
[17:58:20] ✓ Agent created: 0.00s


2025-12-05 17:58:22,756 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2025-12-05 17:58:23,915 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[17:58:34] ✓ Agent execution: 13.87s
[17:58:34] COMPLETE!


#### Format Conversions complexity

1. **PDF** (Most Complex)
    - **Challenges**: Fixed-layout, non-linear text, complex formatting, embedded fonts, scanned documents requiring OCR
    - **Issues**: Text positioning, table detection, header/footer separation, font recognition

    - **pdf library comparison**: 
        - [PDF to markdown 1](https://ai.gopubby.com/benchmarking-pdf-to-markdown-document-converters-fc65a2c73bf2)
        - [PDF to markdown 2](https://ai.gopubby.com/benchmarking-pdf-to-markdown-document-converters-part-2-0439867a3676)
        - [PDF to markdown 3](https://ai.gopubby.com/pdf-to-markdown-conversion-benchmark-part-3-01c22dce8e9d)    

2. **Image (JPEG, PNG, TIFF)**
    - **Challenges**: No inherent text structure, requires OCR, varying image quality
    - **Issues**: OCR accuracy, layout reconstruction, handling of noise and distortions

3. **Word (.docx)**
    - **Challenges**: Complex styling, nested structures, custom formatting, legacy formatting
    - **Issues**: Complex tables, footnotes, cross-references, custom styles

4. **PowerPoint (.pptx)**
    - **Challenges**: Slide-based structure, embedded objects, complex layouts, animations
    - **Issues**: Slide transitions, embedded media, complex formatting

5. **Excel (.xlsx)**
    - **Challenges**: Tabular data, formulas, formatting, merged cells, complex data structures
    - **Issues**: Formula preservation, complex formatting, data type interpretation

6. **HTML**
    - **Challenges**: Irregular structure, inline styles, JavaScript-generated content
    - **Issues**: Inline CSS, malformed HTML, dynamic content

7. **Markdown (.md)**
    - **Challenges**: Already markdown, so conversion is essentially parsing and validation
    - **Issues**: Inconsistent formatting, missing elements

8. **Text (.txt)**
    - **Challenges**: Minimal structure, no formatting
    - **Issues**: Basic text processing, no semantic understanding

8. **Rich Text (.rtf)**
    - **Challenges**: Binary format, complex formatting, embedded objects
    - **Issues**: Formatting inconsistencies, embedded content handling

9. **OpenDocument (.odt)**
    - **Challenges**: XML-based but with complex structures, custom formatting
    - **Issues**: Custom styles, complex document hierarchies

10. **EPUB**
    - **Challenges**: Web-based structure, embedded resources, complex navigation
    - **Issues**: Resource management, complex linking, embedded content